# Reproduce ChakraTS-OS-10M on TIME

Reruns [TIME](https://huggingface.co/spaces/Real-TSF/TIME-leaderboard) for [ChakraTS-OS-10M](https://huggingface.co/yhatlabs/ChakraTS-OS-10M) with the benchmark's own evaluation code, then compares
each result with the submitted one in this folder.

- `MODE = "quick"` (default): two small datasets (`Crypto/D` and `Auto_Production_SF/M`, short term), a few minutes on a CPU.
- `MODE = "full"`: all 98 tasks. Use a GPU: about 19 minutes on one NVIDIA H200.

The model runs on your own machine with [yhatmodels](https://github.com/yhatlabs/yhatmodels). Its first download asks you to accept the model license once with a free yhatlabs.com account; in Colab or on a server, set `YHAT_MODELS_TOKEN` (copy it from https://yhatlabs.com/models/license).

In [ ]:
MODE = "quick"  # "quick" (two small datasets) or "full" (all 98 tasks)
QUICK = ["Crypto/D", "Auto_Production_SF/M"]
TIME_COMMIT = "57a95327043ac5eac637f1ce017f30822105fa93"  # zqiao11/TIME with this model's runner
NAME, RAW = "ChakraTS-OS-10M", "https://raw.githubusercontent.com/yhatlabs/yhatlabs-cookbook/main/benchmarks/time/ChakraTS-OS-10M"

In [ ]:
import os, subprocess, sys, urllib.request
HERE = os.getcwd()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv", "huggingface_hub", "numpy"], check=True)
if not os.path.exists("TIME"):
    subprocess.run(["git", "clone", "-q", "https://github.com/zqiao11/TIME"], check=True)
subprocess.run(["git", "-C", "TIME", "checkout", "-q", TIME_COMMIT], check=True)
from huggingface_hub import snapshot_download
patterns = None if MODE == "full" else [n.split("/")[0] + "/**" for n in QUICK]
os.environ["TIME_DATASET"] = snapshot_download("Real-TSF/TIME", repo_type="dataset", allow_patterns=patterns)

In [ ]:
# The model's license is accepted once per machine: locally the first download opens your browser; in Colab or on a
# server, set YHAT_MODELS_TOKEN (from https://yhatlabs.com/models/license) before running the next cell.

In [ ]:
cmd = ["bash", "scripts/run_chakrats_os.sh", "--output-dir", os.path.join(HERE, "rerun")]
subprocess.run(cmd + (["--dataset", *QUICK, "--terms", "short"] if MODE == "quick" else []), cwd="TIME", check=True)

In [ ]:
import glob
import numpy as np

def submitted_metrics(cfg):
    path = os.path.join(HERE, cfg, "metrics.npz")
    if not os.path.exists(path):  # in Colab: the submitted file from GitHub
        os.makedirs(os.path.dirname(path), exist_ok=True)
        urllib.request.urlretrieve(f"{RAW}/{cfg}/metrics.npz", path)
    return np.load(path)

rows = []
for path in sorted(glob.glob(f"rerun/{NAME}/*/*/*/metrics.npz")):
    cfg = os.path.relpath(os.path.dirname(path), f"rerun/{NAME}")
    new, old = np.load(path), submitted_metrics(cfg)
    rows.append((cfg, np.nanmean(old["MASE"]), np.nanmean(new["MASE"]), np.nanmean(old["CRPS"]), np.nanmean(new["CRPS"]),
                 float(np.nanmax(np.abs(new["MASE"] - old["MASE"])))))
for cfg, m0, m1, c0, c1, d in rows:
    print(f"{cfg:42s} MASE {m0:.4f} -> {m1:.4f}   CRPS {c0:.4f} -> {c1:.4f}   largest per-series MASE change {d:.2e}")